# 04: Route-Hours with Low Demand per Trip
**Milestone 2: Week 3 — Step 7**

This notebook screens for route, day type, and time band combinations with low riders per trip:
- **Screening Criteria:**
  1. `riders_per_trip` in bottom 10% for that day type
  2. Average trips per day $\ge 2.0$
  3. Days of data behind figure $\ge 20$
- **Contextual Enrichment:**
  - Scheduled service-hours and `riders_per_service_hour`
  - Exclusive stop coverage share (stops served by no other route)
  - Top overlapping partner route and overlap share

> **Important Principle:** We explicitly call this list **'low riders per trip'**, never 'inefficient routes'. It is a screening list for operational review, not a verdict. Low patronage often occurs on routes serving essential access without redundancy (e.g., M35 is the only route serving 100% of its stops).

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.eda_utils import typical_days, DAY_TYPE_COLORS, calc_period_ratio, calc_cv

DATA = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
TAB = ROOT / "reports" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

matched = pd.read_parquet(DATA / "matched_hourly.parquet")
profiles = pd.read_parquet(DATA / "route_profiles.parquet")
print(f"Loaded matched hourly: {len(matched):,} rows (41 routes)")
print(f"Loaded route profiles: {len(profiles)} routes")


Loaded matched hourly: 233,227 rows (41 routes)
Loaded route profiles: 41 routes


## Step 7.1: 10th Percentile Thresholds by Day Type
Thresholds are derived from the distribution of route-band riders per trip across all active routes.

In [2]:
perf_table = pd.read_csv(TAB / 'm2_route_performance_table.csv')

thresholds = {}
for dt in ['Typical Weekday', 'Typical Saturday', 'Typical Sunday']:
    sub = perf_table[perf_table['day_type'] == dt].dropna(subset=['riders_per_trip'])
    p10 = sub['riders_per_trip'].quantile(0.10)
    thresholds[dt] = round(p10, 2)
    print(f'{dt:18s} Bottom 10% Threshold: <= {p10:.2f} riders/trip')

pd.Series(thresholds, name='P10_Threshold').to_frame()

Typical Weekday    Bottom 10% Threshold: <= 7.36 riders/trip
Typical Saturday   Bottom 10% Threshold: <= 6.53 riders/trip
Typical Sunday     Bottom 10% Threshold: <= 5.59 riders/trip


,P10_Threshold
Typical Weekday,7.36
Typical Saturday,6.53
Typical Sunday,5.59


## Step 7.2: Operational Review List: Low Riders per Trip
Displays combinations meeting all 3 criteria with stop coverage and corridor overlap context.

In [3]:
low_df = pd.read_csv(TAB / 'm2_low_riders_per_trip.csv')
print(f'Total qualified combinations for review: {len(low_df)}')
print(f'Breakdown by day type: {dict(low_df["day_type"].value_counts())}')

print('\nTop 15 Lowest Riders per Trip Combinations:')
low_df.head(15)

Total qualified combinations for review: 47
Breakdown by day type: {'Typical Weekday': np.int64(17), 'Typical Saturday': np.int64(15), 'Typical Sunday': np.int64(15)}

Top 15 Lowest Riders per Trip Combinations:


,day_type,route,time_band,riders_per_trip,riders_per_service_hour,avg_trips_per_day,days_count,exclusive_stop_share,top_overlap_partner
0,Typical Saturday,M35,night,0.71,3.57,65.6,36,100.0%,None (0.0%)
1,Typical Saturday,M35,morning_peak,1.66,8.63,27.0,36,100.0%,None (0.0%)
2,Typical Saturday,M35,evening_peak,2.40,11.52,41.0,35,100.0%,None (0.0%)
3,Typical Saturday,M35,midday,3.48,17.68,59.3,36,100.0%,None (0.0%)
4,Typical Saturday,M22,night,3.94,10.93,19.9,36,72.1%,M20 (9.3%)
5,Typical Saturday,M8,morning_peak,3.96,13.46,11.0,36,85.7%,M1 (7.1%)
6,Typical Saturday,M20,night,4.16,8.89,65.9,36,51.2%,M7 (18.8%)
7,Typical Saturday,M55,night,4.39,12.32,56.9,36,41.8%,M1 (31.3%)
8,Typical Saturday,M125,night,4.66,12.84,91.8,36,51.2%,M101 (43.9%)
9,Typical Saturday,M55,morning_peak,4.94,14.24,23.0,36,41.8%,M1 (31.3%)


## Step 7.3: Policy Context & Network Role Analysis
A low riders-per-trip figure does not imply a route should be cut. We examine coverage roles:
1. **M35 (Randall's Island Shuttle):** Averages 0.71–3.48 riders/trip, but is the only route serving 100% of its stops (23/23 exclusive stops). It is the sole transit connection to Randall's Island social service and municipal facilities.
2. **M22 / M8 (Crosstown / Lower Manhattan Coverage):** Maintain 72–86% exclusive stops, ensuring access where avenue routes do not reach.
3. **M55 / M20 (Off-Peak Avenue Feeders):** Have parallel alternatives (e.g. M1, M7), making them candidates for schedule coordination rather than route elimination.

In [4]:
coverage = pd.read_csv(TAB / 'm2_route_stop_coverage.csv').set_index('route')

print('Coverage analysis for key low-demand routes:')
for r in ['M35', 'M22', 'M8', 'M55', 'M20', 'M10']:
    ex_share = coverage.loc[r, 'exclusive_stop_share']
    ex_stops = coverage.loc[r, 'exclusive_stops']
    print(f'Route {r:4s}: {ex_share*100:5.1f}% unique stops ({ex_stops} stops served by NO other bus)')

Coverage analysis for key low-demand routes:
Route M35 : 100.0% unique stops (23 stops served by NO other bus)
Route M22 :  72.1% unique stops (31 stops served by NO other bus)
Route M8  :  85.7% unique stops (36 stops served by NO other bus)
Route M55 :  41.8% unique stops (28 stops served by NO other bus)
Route M20 :  51.2% unique stops (41 stops served by NO other bus)
Route M10 :  88.6% unique stops (78 stops served by NO other bus)
